# The Analog Wall, reproducible demo

This notebook reproduces the headline results of the pipeline directly from the
open result tables in this repository. No GPU, no network, no docking rerun: it
reads the saved CSVs (the exact tables behind the paper figures) and recomputes
the numbers so you can check them.

**What it shows**
1. The analog wall: most natural products have no usable chemical analog.
2. Nomination + confidence: pick a molecule, see its ranked targets and where each sits relative to the wall.
3. The resistance screen: which orphans are predicted more resistance-robust than the front-line drug.
4. The carotenoid to PXR convergence, structurally validated by docking.

Data sources are all in `data/` and `figures/`. Run top to bottom.

In [1]:
import pandas as pd, numpy as np
pd.set_option('display.max_colwidth', 40)
# All inputs are the same tables that back the paper figures.
noms = pd.read_csv('data/microbiome_confident_nominations.csv')
ddg  = pd.read_csv('data/screen_ddg_scores.csv')
pxr  = pd.read_csv('data/pxr_dock_final.csv')
print('nominations:', noms.shape, '| ddG rows:', ddg.shape, '| PXR docking:', pxr.shape)

## 1. The analog wall

Every nomination carries `analog_similarity`: the Tanimoto similarity of the query
to its nearest known binder. The analog wall sits at 0.5. This table holds the
**confident** nominations, which are above-wall by construction, so we can see how
far above the wall each sits. In the full library screen (see `PAPER.md`), ~63% of
695k natural products fall *below* this wall and no similarity method can reach
them, which is exactly why the pipeline reports a confidence distance rather than a
bare rank.

In [2]:
wall = 0.5
print(f'confident nominations: {len(noms)}  ({noms.metabolite.nunique()} metabolites, {noms.target_name.nunique()} targets)')
print(f'all confident nominations are above the wall (>=0.5) by construction')
print('similarity distribution above the wall:')
noms.analog_similarity.describe()[['min','25%','50%','75%','max']]

confident nominations: 767  (310 metabolites, 100 targets)
all confident nominations are above the wall (>=0.5) by construction
similarity distribution above the wall:


min    0.500000
25%    0.520000
50%    0.571429
75%    0.645455
max    0.900000

## 2. Nominate targets for a molecule

Pick any metabolite; the pipeline returns its ranked targets with the wall flag.
Here we use *rhodopin*, one of the microbial carotenoids in the flagship cluster.

In [3]:
def nominate(mol):
    r = noms[noms.metabolite == mol].sort_values('analog_similarity', ascending=False).copy()
    r['confidence'] = np.where(r.analog_similarity >= 0.5, 'above wall', 'below wall')
    return r[['target_name','target_class','analog_similarity','confidence']]
nominate('rhodopin')

                                      target_name      target_class  analog_similarity  confidence
86                          Progesterone receptor  Nuclear receptor           0.714286  above wall
89  Nuclear receptor subfamily 1 group I member 2  Nuclear receptor           0.714286  above wall

## 3. Resistance screen: robust orphans per target

For each pathogen target we docked below-wall orphans and the front-line drug into
the wild-type pocket, then scored the affinity change (ΔΔG) across the target's
known pocket mutations. Worst-case ΔΔG = the biggest predicted affinity loss over
those mutations. An orphan is resistance-robust when its worst-case ΔΔG is at or
below the drug's, i.e. it loses less to resistance than the drug does.

In [4]:
d = ddg.dropna(subset=['ddg']).copy()
wc = d.groupby(['target','ligand','kind'])['ddg'].max().reset_index()   # worst-case per ligand
rows = []
for tgt, g in wc.groupby('target'):
    orph = g[g.kind == 'ORPHAN']; drug = g[g.kind.isin(['DRUG','POSITIVE'])]
    if orph.empty or drug.empty: continue
    drug_liability = drug.ddg.max()
    rows.append({'target': tgt.split(' | ')[0],
                 'organism': tgt.split(' | ')[1],
                 'n_orphans': len(orph),
                 'best_orphan_worstcase_ddg': round(orph.ddg.min(), 2),
                 'drug_worstcase_ddg': round(drug_liability, 2),
                 'orphans_beating_drug': int((orph.ddg < drug_liability).sum())})
res = pd.DataFrame(rows).sort_values('orphans_beating_drug', ascending=False)
res

  target                    organism  n_orphans  best_orphan_worstcase_ddg  drug_worstcase_ddg  orphans_beating_drug
3   InhA  Mycobacterium tuberculosis         15                      -0.07                0.18                    10
1   GyrA  Mycobacterium tuberculosis         23                       0.54                0.79                     8
0   DHFR            Escherichia coli         13                       0.23                1.35                     7
2   GyrB  Mycobacterium tuberculosis         15                       0.57                1.15                     4
5   RpoB  Mycobacterium tuberculosis          4                       0.97                1.65                     4
4   KatG  Mycobacterium tuberculosis         15                       0.00                0.06                     1

On DHFR, the enzyme family the ΔΔG scorer was validated on, **7 orphans** sit
below the trimethoprim-class control's worst-case liability; on InhA, **10**.
These are the natural products predicted to keep working as resistance emerges.

## 4. The carotenoid to PXR convergence

Nine structurally related microbial carotenoids all nominated the xenobiotic
receptor PXR. We docked them plus the canonical agonist rifampicin into human PXR
(PDB 1NRL). Every carotenoid lands within 1 kcal/mol of the rifampicin control.

In [5]:
carotenoids = {"7,8-dihydro-beta-carotene","chloroxanthin","dihydroisopentenyldehydrorhodopin",
    "echinenone","1'-hydroxytorulene","4,4'-diaponeurosporenoic acid","rhodopin",
    "(3E)-3,4-didehydrorhodopin","isopentenyldehydrorhodopin"}
control = pxr.loc[pxr.kind == 'POSITIVE', 'affinity'].iloc[0]
carot = pxr[pxr.id.isin(carotenoids)].sort_values('affinity')
print(f'rifampicin control: {control:.2f} kcal/mol')
print(f'carotenoids: {carot.affinity.min():.2f} to {carot.affinity.max():.2f} (median {carot.affinity.median():.2f})')
print(f'all {len(carot)} within 1 kcal/mol of control:', bool((abs(carot.affinity - control) <= 1.0).all()))
carot[['id','affinity']]

rifampicin control: -6.56 kcal/mol
carotenoids: -6.59 to -5.69 (median -6.14)
all 9 within 1 kcal/mol of control: True


                                   id  affinity
1           7,8-dihydro-beta-carotene    -6.586
3                       chloroxanthin    -6.533
4   dihydroisopentenyldehydrorhodopin    -6.280
5                          echinenone    -6.152
6                  1'-hydroxytorulene    -6.140
7       4,4'-diaponeurosporenoic acid    -6.102
8                            rhodopin    -6.017
9          (3E)-3,4-didehydrorhodopin    -5.889
10         isopentenyldehydrorhodopin    -5.686

## Provenance

Every table here is the exact artifact behind the corresponding paper figure.
Results are computational nominations for wet-lab follow-up, not claimed binders.
See `PAPER.md` and `PROJECT_B_PAPER.md` for full methods and caveats.

*Built with Claude.*